In [ ]:
import torch

size = 128

zongyi_dataset = torch.load('../../../2D_NS_old/2D_NS_Zongyi_Li/recurrent/datasets/2D/NS/NS_data_zongyi_test_all_frame.pt', weights_only=False)
me_dataset = torch.load(f'./dim2d_nx{size}_N5_solver=exponax_nu0.000_t20.0_test_all_frames.pt', weights_only=False)

In [8]:
zongyi_dataset["y"].shape

torch.Size([50, 64, 64, 20])

In [10]:
me_dataset["y"].shape

torch.Size([5, 256, 256, 21])

In [11]:
zongyi_dataset["x"].shape

torch.Size([50, 64, 64])

In [12]:
me_dataset["x"].shape

torch.Size([5, 256, 256])

In [18]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import torch

def spectral_upsample(field, target_size=256):
    *batch_dims, H, W = field.shape
    assert target_size >= H and target_size >= W, "目标尺寸必须大于输入尺寸"
    freq = torch.fft.fft2(field, norm='ortho')
    freq_shifted = torch.fft.fftshift(freq, dim=(-2, -1))
    pad_H = (target_size - H) // 2
    pad_W = (target_size - W) // 2
    new_freq_shifted = torch.zeros(
        *batch_dims, target_size, target_size, 
        dtype=freq_shifted.dtype, device=freq_shifted.device
    )
    start_H = pad_H
    start_W = pad_W
    new_freq_shifted[..., start_H:start_H+H, start_W:start_W+W] = freq_shifted
    new_freq = torch.fft.ifftshift(new_freq_shifted, dim=(-2, -1))
    upsampled = torch.fft.ifft2(new_freq, norm='ortho')
    return (target_size / H) * upsampled.real

def create_heatmap_gif(array1, array2, 
                      title1='Heatmap 1', title2='Heatmap 2', 
                      cmap='viridis', figsize=(18, 6),
                      colorbar=True, 
                      fps=10, dpi=100, text="training"):
    # Create temporary directory for frames
    os.makedirs('temp_frames', exist_ok=True)
    
    frame_files = []
    
    for idx in range(array1.shape[0]):
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=figsize)
        
        # Plot first heatmap
        im1 = ax1.imshow(array1[idx], cmap=cmap, origin='lower')
        ax1.set_title(title1)
        ax1.set_xlabel('X')
        ax1.set_ylabel('Y')
        if colorbar:
            fig.colorbar(im1, ax=ax1, label='Value')
        
        # Plot second heatmap
        im2 = ax2.imshow(array2[idx], cmap=cmap, origin='lower')
        ax2.set_title(title2)
        ax2.set_xlabel('X')
        ax2.set_ylabel('Y')
        if colorbar:
            fig.colorbar(im2, ax=ax2, label='Value')

        # Calculate difference and statistics
        resized = spectral_upsample(array1[idx], target_size=array2[idx].shape[0])
        difference = array2[idx] - resized
        abs_diff = torch.abs(difference)
        diff_mean = torch.mean(abs_diff)
        diff_max = torch.max(abs_diff)
        # diff_min = torch.min(abs_diff)
        # diff_median = torch.median(abs_diff)
        # diff_std = torch.std(abs_diff)
        
        # Plot difference heatmap with statistics in title
        im3 = ax3.imshow(difference, cmap="coolwarm", origin='lower')
        stats_text = (
                     # f"Abs Diff Stats:\n"
                     f"\nMean: {diff_mean:.2e} "
                     f"Max: {diff_max:.2e} "
                     # f"Min: {diff_min:.2e}\n"
                     # f"Median: {diff_median:.2e}\n"
                     # f"Std: {diff_std:.2e}"
                     )
        ax3.set_title(f"{title2}-{title1} difference {stats_text}")
        ax3.set_xlabel('X')
        ax3.set_ylabel('Y')
        if colorbar:
            fig.colorbar(im3, ax=ax3, label='Difference Value')
        
        plt.suptitle(f"Navier-Stokes Vorticity Transport (frame {idx+1}) ({text})")
        plt.tight_layout()
        
        # Save frame
        frame_file = f"temp_frames/frame_{idx:04d}.png"
        plt.savefig(frame_file, dpi=dpi, bbox_inches='tight')
        frame_files.append(frame_file)
        plt.close()
    
    # Create GIF from frames
    images = [Image.open(f) for f in frame_files]
    os.makedirs(f"gifs_nx{size}", exist_ok=True)
    images[0].save(f"gifs_nx{size}/{text}.gif", 
                  save_all=True, 
                  append_images=images[1:], 
                  duration=1000//fps, 
                  loop=0)
    
    # Clean up temporary files
    for f in frame_files:
        os.remove(f)
    os.rmdir('temp_frames')
    
    print(f"GIF saved as {text}.gif")

In [19]:
nt = 20
for idx in [0,1,2,3,4]:
    create_heatmap_gif(
        zongyi_dataset["y"][idx].permute(2,1,0), 
        me_dataset["y"][idx].permute(2,1,0)[:-1], 
        title1='Zongyi Li 64*64',
        title2=f'Me {size}*{size}',
        fps=5,
        text=f"Zongyi Li and Me (exponax) comparison size={size} nt={nt} idx={idx}"
    )

GIF saved as Zongyi Li and Me (exponax) comparison size=64 nt=20 idx=0.gif
GIF saved as Zongyi Li and Me (exponax) comparison size=64 nt=20 idx=1.gif
GIF saved as Zongyi Li and Me (exponax) comparison size=64 nt=20 idx=2.gif
GIF saved as Zongyi Li and Me (exponax) comparison size=64 nt=20 idx=3.gif
GIF saved as Zongyi Li and Me (exponax) comparison size=64 nt=20 idx=4.gif


In [16]:
zongyi_dataset["y"].shape

torch.Size([50, 64, 64, 20])

In [17]:
me_dataset["y"].shape

torch.Size([5, 256, 256, 21])

In [ ]:
me_dataset["y"].shape

In [23]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

PyTorch version: 2.7.0+cu126
CUDA version: 12.6


In [24]:
import torch
print(torch._C._cuda_getDeviceProperties(0).major, torch._C._cuda_getDeviceProperties(0).minor)

AttributeError: module 'torch._C' has no attribute '_cuda_getDeviceProperties'

In [25]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
name = torch.cuda.get_device_name(device)
capability = torch.cuda.get_device_capability(device)

print(f"Device name: {name}")
print(f"Compute capability (sm): {capability[0]}.{capability[1]}")

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx